In [71]:
import sys
import os
# Get the current directory and parent directory
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)
sys.path.insert(0, current_dir)

# Import from the-machine module
import importlib.util
the_machine_path = os.path.join(parent_dir, "the-machine")
sys.path.insert(0, the_machine_path)

# Import all from the-machine and custom_matroid_functions
from HM_module import *
from custom_matroid_functions import *

In [72]:
P = LaurentPolynomialRing(ZZ, 'x')
x = P.gen()
k, n = 5, 5
#M = matroids.CompleteGraphic(n)
M = matroids.Uniform(k, n)
#M = matroids.catalog.Fano()
L = M.lattice_of_flats()
r = M.rank()
rank_func = lambda x : M.rank(x)

#e = next(x for x in M.groundset() if x  not in M.coloops())
e = list(M.groundset())[0]
print(M.groundset())
print("deleting ", e)
mdele = M.delete(e)
Eq_M = create_deformed_algebra(M, base_ring=P)
Eq_mdele = create_deformed_algebra(mdele, base_ring=P)
E_mdele = Eq_mdele.natural()

E = Eq_M.natural()
E_basis = E.basis()
E_mdele_basis = E_mdele.basis()

C = Eq_M.C()
C_dele = Eq_mdele.C()
C_basis = C.basis()
C_dele_basis = C_dele.basis()

flats_keys = list(E_basis.keys())

chow = chow_polynomial
chibar = reduced_characteristic_polynomial
char_poly = characteristic_polynomial
kl_poly = lambda M: M.lattice_of_flats().kazhdan_lusztig_polynomial()(x)
grev = lambda M: P(x**M.rank())
mu_rev = lambda M: P(x**M.rank() * char_poly(M)(0))
mu = lambda M: P(char_poly(M)(0))

frozenset({0, 1, 2, 3, 4})
deleting  0


In [73]:
def flat_division(M, e):
    L = M.lattice_of_flats()
    flat_div = {'S':set(), 'T':set(), 'A':set(), 'B':set()}
    for F in L:
        if e in F:
            if F - {e} in L:
                flat_div['T'].add(F)
            else:
                flat_div['A'].add(F)
        else:
            if F.union({e}) in L:
                flat_div['S'].add(F)
            else:
                flat_div['B'].add(F)
    return flat_div

def get_pal0_part(f):
    """
    Extracts the Pal(0) part of a Laurent polynomial f.
    """
    if f == 0: return f
    x_var = f.parent().gen()
    f_inv = f.subs({x_var: x_var**-1})
    numerator = (x_var**2) * f_inv - f
    denominator = (x_var**2) - 1
    q, r = numerator.quo_rem(denominator)
    return q


flat_div = flat_division(M, e)
#flat_div['S'].remove(frozenset())

def convolve(f, g):
    return lambda M: sum([f(M.delete(M.groundset() - F))(x) * g(M.contract(F))(x) for F in list(M.lattice_of_flats())])

conv = lambda f, g: convolve(f, g)(M)

def rev(f):
    return lambda M: f(M)(x**-1) * x**M.rank()

rightAugpoly = convolve(chow, rev(kl_poly))
bbasepoly = convolve(mu_rev, rightAugpoly)
zeta = lambda M: P(1) if M else 0
klrev = rev(kl_poly)
leftAugpoly = convolve(grev, chow_polynomial)

In [74]:
def get_matroid_key(M):
    rank = M.rank()
    size = len(M.groundset())
    try:
        can = M.canonical_form()
        return (rank, size, str(can))
    except AttributeError:
        bases = tuple(sorted(tuple(sorted(b)) for b in M.bases()))
        return (rank, size, str(bases))

def convolution_inverse(f):
    cache_inv = {}
    
    def f_inv(M):
        m_key = get_matroid_key(M)
        if m_key in cache_inv:
            return cache_inv[m_key]
        
        if M.rank() == 0 and M.size() == 0:
            res = 1 / f(M)
            cache_inv[m_key] = res
            return res
        
        # f_inv(M) = - (1/f(empty)) * sum_{empty != F <= E} f(M|F) f_inv(M/F)
        empty_matroid = matroids.Uniform(0, 0)
        f0 = f(empty_matroid)
        
        L = M.lattice_of_flats()
        E = M.groundset()
        
        total = P(0)
        for F in L:
            if len(F) == 0: continue
            restriction = M.delete(list(set(E) - set(F)))
            contraction = M.contract(list(F))
            total += f(restriction) * f_inv(contraction)
            
        res = -total / f0
        cache_inv[m_key] = res
        return res
    
    return f_inv

In [75]:
for F in flat_div['S']:
    print(F)

frozenset({3, 4})
frozenset({2})
frozenset({1, 4})
frozenset({2, 3, 4})
frozenset({2, 3})
frozenset({1, 2, 4})
frozenset({1, 2})
frozenset({1, 2, 3})
frozenset({3})
frozenset({2, 4})
frozenset({1})
frozenset({1, 2, 3, 4})
frozenset()
frozenset({1, 3})
frozenset({1, 3, 4})
frozenset({4})


In [76]:
Finv = convolution_inverse(rightAugpoly)
inv = convolve(Finv, grev)

In [77]:
def deleconj(M, e, f):
    E = M.groundset()
    mdele = M.delete(e)
    og_matroid_poly = f(M)
    deleted_matroid_poly = f(mdele)
    bbasecoeff = lambda M: bbasepoly(M)(x^(-2))
    sum1 = 0
    for G in flat_div['S']:
        sum2 = 0
        restricted_filter_G = [F for F in L.order_filter([G]) if F in flat_div['S']]
        print("flat:      ", G)
        for F in restricted_filter_G:
            poly1 = x^(-2) * bbasecoeff(M.contract(F.union({e})))
            poly2 = bbasecoeff(M.contract(F))
            poly3 = bbasecoeff(mdele.contract(F))
            print("     F <= G", F)
            print(" poly1, poly2, poly3: ", poly1,",", poly2,",", poly3)
            term1 = x^(M.rank()- M.rank(G)) * (poly1 + poly2 - poly3)
            term2 = inv(M.delete(E - F).contract(G))(x^(-2))
            print("term1,   term2:", term1, ",", term2)
            print("the product   :", term1 * term2)
            sum2 += term1 * term2
        term3 = x^(M.rank(G)) * rightAugpoly(M.delete(E - G))(x^-2)
        sum1 += sum2 * term3
        print("inner sum: ", sum2)
        print("term3      ",term3)
        print("product    ", sum2 * term3)
        print("--------------------------------")
        print("********************************")
    print("summation equals:", sum1)
    print("f(M):     ", og_matroid_poly)
    print("f(M\\e):  ", deleted_matroid_poly)
    return P(sum1)

right = deleconj(M, e, rightAugpoly)
print("right", right)

flat:       frozenset({3, 4})
     F <= G frozenset({3, 4})
 poly1, poly2, poly3:  x^-4 + x^-2 , x^-4 + 4*x^-2 + 1 , x^-2 + 1
term1,   term2: 2*x^-1 + 4*x , 1
the product   : 2*x^-1 + 4*x
     F <= G frozenset({1, 3, 4})
 poly1, poly2, poly3:  x^-2 , x^-2 + 1 , 1
term1,   term2: 2*x , -1
the product   : -2*x
     F <= G frozenset({2, 3, 4})
 poly1, poly2, poly3:  x^-2 , x^-2 + 1 , 1
term1,   term2: 2*x , -1
the product   : -2*x
     F <= G frozenset({1, 2, 3, 4})
 poly1, poly2, poly3:  x^-2 , 1 , 1
term1,   term2: x , -x^-2 + 1
the product   : -x^-1 + x
inner sum:  x^-1 + x
term3       x^-2 + 3 + x^2
product     x^-3 + 4*x^-1 + 4*x + x^3
--------------------------------
********************************
flat:       frozenset({2})
     F <= G frozenset({2})
 poly1, poly2, poly3:  x^-6 + 4*x^-4 + x^-2 , x^-6 + 11*x^-4 + 11*x^-2 + 1 , x^-4 + 4*x^-2 + 1
term1,   term2: 2*x^-2 + 14 + 8*x^2 , 1
the product   : 2*x^-2 + 14 + 8*x^2
     F <= G frozenset({1, 2})
 poly1, poly2, poly3:  x^-4 + x^-

In [78]:
lhs = x^(M.rank()) * (rightAugpoly(M)(x^-2) - rightAugpoly(mdele)(x^-2))
print(lhs)


x^-5 + 30*x^-3 + 116*x^-1 + 98*x + 16*x^3


In [79]:
# Compute the pal0 part of the RHS expression
m = M.rank()

for F in flat_div['S']:
    f = M.rank(F)
    
    M_F_cup_e = M.contract(F.union({e}))
    K1 = bbasepoly(M_F_cup_e)
    
    M_F = M.contract(F)
    K2 = bbasepoly(M_F)
    
    M_e_mod_F = M.delete(e).contract(F)
    K3 = bbasepoly(M_e_mod_F)
    
    # Evaluate at x^-2
    K1_eval = K1.subs({x: x**-2})
    K2_eval = K2.subs({x: x**-2})
    K3_eval = K3.subs({x: x**-2})
    
    # Combine terms
    term_poly = x**(m - f) * (x**-2 * K1_eval + K2_eval - K3_eval)
    pal0 = get_pal0_part(term_poly)
    
    print(f"F = {F}")
    print(f"  term_poly = {term_poly}")
    print(f"  pal0 = {pal0}")
    print("-" * 50)


F = frozenset({3, 4})
  term_poly = 2*x^-1 + 4*x
  pal0 = 2*x^-1 + 2*x
--------------------------------------------------
F = frozenset({2})
  term_poly = 2*x^-2 + 14 + 8*x^2
  pal0 = 2*x^-2 + 8 + 2*x^2
--------------------------------------------------
F = frozenset({1, 4})
  term_poly = 2*x^-1 + 4*x
  pal0 = 2*x^-1 + 2*x
--------------------------------------------------
F = frozenset({2, 3, 4})
  term_poly = 2
  pal0 = 2
--------------------------------------------------
F = frozenset({2, 3})
  term_poly = 2*x^-1 + 4*x
  pal0 = 2*x^-1 + 2*x
--------------------------------------------------
F = frozenset({1, 2, 4})
  term_poly = 2
  pal0 = 2
--------------------------------------------------
F = frozenset({1, 2})
  term_poly = 2*x^-1 + 4*x
  pal0 = 2*x^-1 + 2*x
--------------------------------------------------
F = frozenset({1, 2, 3})
  term_poly = 2
  pal0 = 2
--------------------------------------------------
F = frozenset({3})
  term_poly = 2*x^-2 + 14 + 8*x^2
  pal0 = 2*x^-2 + 

In [80]:
def deleconj2(M, e, f, bbasepoly, x, P):
    E = set(M.groundset())
    mdele = M.delete(e)
    og_matroid_poly = f(M)
    deleted_matroid_poly = f(mdele)
    
    bbasecoeff = lambda mat: bbasepoly(mat)(x**(-2))
    
    sum1 = 0
    
    # Single loop over F
    for F in flat_div['S']:
        F_set = set(F)
        F_union_e = list(F_set.union({e}))
        
        poly1 = x**(-2) * bbasecoeff(M.contract(F_union_e))
        poly2 = bbasecoeff(M.contract(F))
        poly3 = bbasecoeff(mdele.contract(F))
        
        term1 = (poly1 + poly2 - poly3)
        
        # The entire G-loop collapses to evaluating grev on the restriction M^F
        # M.delete(E \ F) is the restriction of M to F
        term_g = x**(-2 * M.rank(F))

        print("flat:", F)
        print("poly1, poly2, poly3:", poly1, ",", poly2, ",", poly3)
        print("term1, term_g:", term1, ",", term_g)
        
        # Accumulate the factored sum
        sum1 += term1 * term_g
        print("term1 * term_g:", term1 * term_g)
        print("------------------")
        print("sum1:", sum1)
        print("------------------")

    print("summation equals:", sum1)
    
    return P(x**(M.rank()) * sum1)

deleconj2(M, e, rightAugpoly, bbasepoly, x, P)

flat: frozenset({3, 4})
poly1, poly2, poly3: x^-4 + x^-2 , x^-4 + 4*x^-2 + 1 , x^-2 + 1
term1, term_g: 2*x^-4 + 4*x^-2 , x^-4
term1 * term_g: 2*x^-8 + 4*x^-6
------------------
sum1: 2*x^-8 + 4*x^-6
------------------
flat: frozenset({2})
poly1, poly2, poly3: x^-6 + 4*x^-4 + x^-2 , x^-6 + 11*x^-4 + 11*x^-2 + 1 , x^-4 + 4*x^-2 + 1
term1, term_g: 2*x^-6 + 14*x^-4 + 8*x^-2 , x^-2
term1 * term_g: 2*x^-8 + 14*x^-6 + 8*x^-4
------------------
sum1: 4*x^-8 + 18*x^-6 + 8*x^-4
------------------
flat: frozenset({1, 4})
poly1, poly2, poly3: x^-4 + x^-2 , x^-4 + 4*x^-2 + 1 , x^-2 + 1
term1, term_g: 2*x^-4 + 4*x^-2 , x^-4
term1 * term_g: 2*x^-8 + 4*x^-6
------------------
sum1: 6*x^-8 + 22*x^-6 + 8*x^-4
------------------
flat: frozenset({2, 3, 4})
poly1, poly2, poly3: x^-2 , x^-2 + 1 , 1
term1, term_g: 2*x^-2 , x^-6
term1 * term_g: 2*x^-8
------------------
sum1: 8*x^-8 + 22*x^-6 + 8*x^-4
------------------
flat: frozenset({2, 3})
poly1, poly2, poly3: x^-4 + x^-2 , x^-4 + 4*x^-2 + 1 , x^-2 + 1
te

x^-5 + 30*x^-3 + 116*x^-1 + 98*x + 16*x^3